In [3]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [4]:
from grb.io import read_data, filter_data
from grb.extinction import correct_host_galaxy_extinction

In [5]:
df = read_data("circular", correct_galactic_extinction=True, add_converted_flux=True)
filtered_df = filter_data(df, filter_name="Ic")
corrected_df = correct_host_galaxy_extinction(filtered_df, 0.126)

In [6]:
from grb.modeling import add_observation
from VegasAfterglow import ObsData, Setups, ParamDef, Scale, Fitter


In [7]:
df_xrt = read_data("xrt")

In [8]:
obsdata = ObsData()
obsdata = add_observation(corrected_df, obsdata=obsdata, input_type="flux_density")
obsdata = add_observation(df_xrt, obsdata=obsdata, input_type="flux")

In [9]:
from grb.const import D_L, REDSHIFT

In [10]:
cfg = Setups()

# Source properties
cfg.lumi_dist = D_L
cfg.z = REDSHIFT              # Redshift

# Model selection (see sections below for all options)
cfg.medium = "wind"       # Ambient medium type
cfg.jet = "tophat"      # Jet structure type

# Physics options
cfg.rvs_shock = True      # Include reverse shock
cfg.fwd_ssc = True        # Forward shock inverse Compton
cfg.rvs_ssc = False       # Reverse shock inverse Compton
cfg.ssc_cooling = True     # IC cooling effects
cfg.kn = True             # Klein-Nishina corrections
cfg.magnetar = False       # Magnetar energy injection

# Numerical parameters
cfg.rtol = 1e-5           # Numerical tolerance

# Basic parameter set
params = [
    ParamDef("E_iso",   1e50,  1e54,  Scale.LOG),     # Isotropic energy in erg
    ParamDef("Gamma0",    10,   500,  Scale.LOG),     # Lorentz factor
    ParamDef("theta_c", 0.01,   0.5,  Scale.LINEAR),  # Opening angle in radians
    ParamDef("theta_v",    0,     0,  Scale.FIXED),   # Viewing angle (on-axis) in radians
    ParamDef("A_star",  1e-3,   1.0,  Scale.LOG), 
    ParamDef("p",        2.1,   2.8,  Scale.LINEAR),  # Electron spectral index
    ParamDef("eps_e",   1e-3,   0.5,  Scale.LOG),     # Electron energy fraction
    ParamDef("eps_B",   1e-5,   0.1,  Scale.LOG),     # Magnetic energy fraction
    ParamDef("xi_e",     0.1,   1.0,  Scale.LINEAR),  # Fraction of accelerated electrons
    ParamDef("tau",        1,   100,  Scale.LOG),
        # Forward + reverse microphysics
    ParamDef("p_r",      2.1,   2.8,  Scale.LINEAR),
    ParamDef("eps_e_r", 1e-3,   0.5,  Scale.LOG),
    ParamDef("eps_B_r", 1e-5,   0.1,  Scale.LOG),
    ParamDef("xi_e_r",   0.1,   1.0,  Scale.LINEAR),
]


In [ ]:
fitter = Fitter(obsdata, cfg)

: 

In [ ]:
result = fitter.fit(
    params,
    resolution=(0.15, 0.5, 10),     # Grid resolution (phi, theta, t)
    sampler="dynesty",             # Nested sampling algorithm
    nlive=1000,                    # Number of live points
    walks=100,                     # Number of random walks per live point
    dlogz=0.5,                     # Stopping criterion (evidence tolerance)
    npool=8,                       # Number of parallel processes
    top_k=10,                      # Number of best-fit parameters to return
)


18:14 bilby INFO    : Running for label 'afterglow', output will be saved to 'bilby_output'


18:14 bilby INFO    : Analysis priors:
18:14 bilby INFO    : log10_E_iso=Uniform(minimum=50.0, maximum=54.0, name='log10_E_iso', latex_label='$\\log_{10}(E_{\\rm iso})$', unit=None, boundary=None)
18:14 bilby INFO    : log10_Gamma0=Uniform(minimum=1.0, maximum=2.6989700043360187, name='log10_Gamma0', latex_label='$\\log_{10}(\\Gamma_0)$', unit=None, boundary=None)
18:14 bilby INFO    : theta_c=Uniform(minimum=0.01, maximum=0.5, name='theta_c', latex_label='$\\theta_c$', unit=None, boundary=None)
18:14 bilby INFO    : log10_A_star=Uniform(minimum=-3.0, maximum=0.0, name='log10_A_star', latex_label='$\\log_{10}(A_*)$', unit=None, boundary=None)
18:14 bilby INFO    : p=Uniform(minimum=2.1, maximum=2.8, name='p', latex_label='$p$', unit=None, boundary=None)
18:14 bilby INFO    : log10_eps_e=Uniform(minimum=-3.0, maximum=-0.3010299956639812, name='log10_eps_e', latex_label='$\\log_{10}(\\epsilon_e)$', unit=None, boundary=None)
18:14 bilby INFO    : log10_eps_B=Uniform(minimum=-5.0, maximum=